# Spark MLlib Taxi Demand Demo

This notebook is the presentation and diagnostics layer for the GCP-trained taxi-demand model. It can read the modeling dataset from either local Parquet output or the GCS-backed cloud Iceberg table, rebuild feature rows, load downloaded GCP model artifacts for holdout diagnostics, and export precomputed predictions for Streamlit.

**Goal:** predict `trip_count` for a pickup zone and time. The notebook uses GCP-trained artifacts for 2025 diagnostics, then refits a simple local LinearRegression model on 2023-2025 for the 2026 Streamlit forecast export.


## Demo Outline

1. Start Spark and choose the data source.
2. Load the ML dataset from local Parquet or cloud Iceberg.
3. Build the feature rows expected by the saved GCP model.
4. Prepare the 2025 holdout rows.
5. Read the downloaded GCP training artifacts.
6. Compare model metrics from GCP.
7. Load the best saved GCP pipeline model.
8. Check actual versus predicted demand on the 2025 holdout set.
9. Refit on 2023-2025 and export Streamlit-ready 2026 predictions.
10. Stop Spark.

Use `DATA_SOURCE = "local_parquet"` for local/offline runs. Use `DATA_SOURCE = "cloud_iceberg"` when running on Dataproc or another Spark environment with access to the GCS Iceberg warehouse.


## 1. Start Spark

Choose one data source:

- `local_parquet`: reads the local Parquet export of `taxi_demand_ml`.
- `cloud_iceberg`: reads `nyc.taxi_demand_ml` from the GCS-backed Iceberg warehouse.

If Spark was already started with different settings, restart the notebook kernel before running this cell again.


In [1]:
from pathlib import Path
import gc
import json
import os
import zipfile

import pandas as pd
from pyspark import SparkContext
from pyspark.ml import PipelineModel
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel

RUNNING_IN_DOCKER = Path("/workspace").exists()

# DATA_SOURCE = "local_parquet"
DATA_SOURCE = "cloud_iceberg"

DEMO_MODE = False
DEMO_DATE_STRIDE_DAYS = 14

SERVING_PREDICTION_YEAR = 2025
SERVING_PREDICTION_MONTH = 1

# Use spark://spark-master:7077 inside the Spark container.
# Use spark://localhost:7077 if running this notebook directly from the Mac host.
SPARK_MASTER = "spark://spark-master:7077"
SHUFFLE_PARTITIONS = "4"
CACHE_STORAGE_LEVEL = StorageLevel.DISK_ONLY

PLOT_SAMPLE_FRACTION = 0.02
PLOT_SAMPLE_LIMIT = 5000

GCP_TRAINING_ROOT = Path("/workspace/output" if RUNNING_IN_DOCKER else "output")
GCP_RUN_ID = "20260508T091500Z-saveall-full"

ICEBERG_PACKAGE = "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2"
GCS_CONNECTOR_JAR = "https://repo1.maven.org/maven2/com/google/cloud/bigdataoss/gcs-connector/hadoop3-2.2.23/gcs-connector-hadoop3-2.2.23-shaded.jar"
LOCAL_PARQUET_PATH = (
    "/workspace/output/taxi_demand_ml_parquet"
    if RUNNING_IN_DOCKER
    else "output/taxi_demand_ml_parquet"
)
LOCAL_PARQUET_ZIP_PATH = f"{LOCAL_PARQUET_PATH}.zip"
CLOUD_ICEBERG_WAREHOUSE = "gs://sparkforgroup-data228-taxi/iceberg/warehouse"
ICEBERG_CATALOG = "nyc"
ML_TABLE = f"{ICEBERG_CATALOG}.taxi_demand_ml"
ZONE_LOOKUP_PATH = "/workspace/data/taxi_zone_lookup.csv" if RUNNING_IN_DOCKER else "data/taxi_zone_lookup.csv"

if DATA_SOURCE not in {"local_parquet", "cloud_iceberg"}:
    raise ValueError('DATA_SOURCE must be "local_parquet" or "cloud_iceberg"')

if DATA_SOURCE == "cloud_iceberg":
    os.environ["PYSPARK_SUBMIT_ARGS"] = f"--packages {ICEBERG_PACKAGE} --jars {GCS_CONNECTOR_JAR} pyspark-shell"
else:
    os.environ.pop("PYSPARK_SUBMIT_ARGS", None)

active_session = SparkSession.getActiveSession()
if active_session is not None:
    active_session.stop()
elif SparkContext._active_spark_context is not None:
    SparkContext._active_spark_context.stop()

spark_builder = (
    SparkSession.builder
    .appName("Spark MLlib Taxi Demand GCP Diagnostics")
    .master(SPARK_MASTER)
    .config("spark.sql.shuffle.partitions", SHUFFLE_PARTITIONS)
    .config("spark.default.parallelism", SHUFFLE_PARTITIONS)
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.python.worker.reuse", "false")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.executor.cores", "2")
)

if DATA_SOURCE == "cloud_iceberg":
    spark_builder = (
        spark_builder
        .config("spark.jars.packages", ICEBERG_PACKAGE)
        .config("spark.jars", GCS_CONNECTOR_JAR)
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG}", "org.apache.iceberg.spark.SparkCatalog")
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG}.type", "hadoop")
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG}.warehouse", CLOUD_ICEBERG_WAREHOUSE)
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
        .config("spark.hadoop.fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
    )

spark = spark_builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Data source: {DATA_SOURCE}")
print(f"Demo mode: {DEMO_MODE}")
print(f"Spark master: {SPARK_MASTER}")
print(f"Shuffle partitions: {SHUFFLE_PARTITIONS}")
print(f"GCP training root: {GCP_TRAINING_ROOT}")
print(f"Streamlit prediction month: {SERVING_PREDICTION_YEAR}-{SERVING_PREDICTION_MONTH:02d}")
if DATA_SOURCE == "local_parquet":
    print(f"Local Parquet path: {LOCAL_PARQUET_PATH}")
else:
    print(f"Cloud Iceberg warehouse: {CLOUD_ICEBERG_WAREHOUSE}")
    print(f"Cloud Iceberg table: {ML_TABLE}")


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/spark/.ivy2/cache
The jars for the packages stored in: /home/spark/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
com.google.cloud.bigdataoss#gcs-connector added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-809e0496-9445-48aa-a20a-a71c24781e0d;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.2 in central
	found com.google.cloud.bigdataoss#gcs-connector;hadoop3-2.2.23 in central
	found com.google.api-client#google-api-client-jackson2;2.0.1 in central
	found com.google.api-client#google-api-client;2.0.1 in central
	found com.google.oauth-client#google-oauth-client;1.34.1 in central
	found com.google.http-client#google-http-client;1.42.3 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.15 in central
	found commons-logging#commons-logging;1.2 in central
	found commons-codec#commons-co

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark version: 3.5.3
Data source: cloud_iceberg
Demo mode: False
Spark master: spark://spark-master:7077
Shuffle partitions: 4
GCP training root: /workspace/output
Streamlit prediction month: 2025-01
Cloud Iceberg warehouse: gs://sparkforgroup-data228-taxi/iceberg/warehouse
Cloud Iceberg table: nyc.taxi_demand_ml


## 2. Load the ML Dataset

`local_parquet` reads the local Parquet export of `taxi_demand_ml`. `cloud_iceberg` reads the rebuilt GCS-backed Iceberg table `nyc.taxi_demand_ml`. Both modes produce the same `raw_df` shape for the rest of the notebook.


In [2]:
required_columns = [
    "PULocationID",
    "pickup_year",
    "pickup_month",
    "pickup_day_of_week",
    "pickup_hour",
    "is_weekend",
    "trip_count",
]

if DATA_SOURCE == "local_parquet":
    parquet_path = Path(LOCAL_PARQUET_PATH)
    parquet_zip_path = Path(LOCAL_PARQUET_ZIP_PATH)
    if not parquet_path.exists() and parquet_zip_path.exists():
        print(f"Extracting {parquet_zip_path} to {parquet_path.parent}")
        with zipfile.ZipFile(parquet_zip_path, "r") as archive:
            archive.extractall(parquet_path.parent)
    raw_df = spark.read.parquet(LOCAL_PARQUET_PATH)
    input_description = LOCAL_PARQUET_PATH
elif DATA_SOURCE == "cloud_iceberg":
    raw_df = spark.read.table(ML_TABLE)
    input_description = ML_TABLE
else:
    raise ValueError('DATA_SOURCE must be "local_parquet" or "cloud_iceberg"')

missing_columns = [column for column in required_columns if column not in raw_df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

if "pickup_date" not in raw_df.columns and "pickup_day" not in raw_df.columns:
    raise ValueError("The notebook needs pickup_date or pickup_day to build monthly prediction features.")

if raw_df.limit(1).count() == 0:
    raise ValueError(f"No rows found in {input_description}")

zone_lookup_df = (
    spark.read
    .option("header", True)
    .csv(ZONE_LOOKUP_PATH)
    .select(
        F.col("LocationID").cast("long").alias("PULocationID"),
        F.col("Borough").alias("borough"),
        F.col("Zone").alias("zone"),
        F.col("service_zone").alias("service_zone"),
    )
)

print(f"Loaded dataset from {input_description}")
print(f"Input partitions: {raw_df.rdd.getNumPartitions()}")
raw_df.printSchema()
raw_df.select([column for column in required_columns if column in raw_df.columns]).show(5, truncate=False)


Py4JJavaError: An error occurred while calling o57.table.
: java.lang.RuntimeException: java.lang.reflect.InvocationTargetException
	at org.apache.hadoop.util.ReflectionUtils.newInstance(ReflectionUtils.java:137)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3467)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:174)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3521)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:540)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:365)
	at org.apache.iceberg.hadoop.Util.getFs(Util.java:56)
	at org.apache.iceberg.hadoop.HadoopCatalog.initialize(HadoopCatalog.java:112)
	at org.apache.iceberg.CatalogUtil.loadCatalog(CatalogUtil.java:255)
	at org.apache.iceberg.CatalogUtil.buildIcebergCatalog(CatalogUtil.java:309)
	at org.apache.iceberg.spark.SparkCatalog.buildIcebergCatalog(SparkCatalog.java:154)
	at org.apache.

## 3. Build Model Feature Rows

The saved GCP pipeline expects one row per pickup location, date, and hour with the historical-average feature columns used during training. This step combines source rows, fills missing location/date/hour combinations with zero demand, and calculates those historical features.


In [ ]:
def date_from_year_month_day(year_col, month_col, day_col):
    return F.to_date(
        F.concat_ws(
            "-",
            year_col.cast("string"),
            F.format_string("%02d", month_col),
            F.format_string("%02d", day_col),
        )
    )


def build_source_agnostic_demand(input_df):
    select_columns = [
        F.col("PULocationID").cast("long").alias("PULocationID"),
        F.col("pickup_year").cast("int").alias("pickup_year"),
        F.col("pickup_month").cast("int").alias("pickup_month"),
        F.col("pickup_day_of_week").cast("int").alias("pickup_day_of_week"),
        F.col("pickup_hour").cast("int").alias("pickup_hour"),
        F.col("is_weekend").cast("boolean").alias("is_weekend"),
        F.col("trip_count").cast("double").alias("trip_count"),
    ]

    if "pickup_date" in input_df.columns:
        select_columns.append(F.to_date("pickup_date").alias("pickup_date"))
    if "pickup_day" in input_df.columns:
        select_columns.append(F.col("pickup_day").cast("int").alias("pickup_day"))

    base_df = input_df.select(*select_columns)

    if "pickup_day" not in base_df.columns:
        base_df = base_df.withColumn("pickup_day", F.dayofmonth("pickup_date"))
    if "pickup_date" not in base_df.columns:
        base_df = base_df.withColumn(
            "pickup_date",
            date_from_year_month_day(F.col("pickup_year"), F.col("pickup_month"), F.col("pickup_day")),
        )

    return (
        base_df
        .dropna(subset=[
            "PULocationID",
            "pickup_date",
            "pickup_year",
            "pickup_month",
            "pickup_day",
            "pickup_day_of_week",
            "pickup_hour",
            "is_weekend",
            "trip_count",
        ])
        .groupBy(
            "PULocationID",
            "pickup_date",
            "pickup_year",
            "pickup_month",
            "pickup_day",
            "pickup_day_of_week",
            "pickup_hour",
            "is_weekend",
        )
        .agg(F.sum("trip_count").alias("trip_count"))
        .withColumn("month_index", F.col("pickup_year") * F.lit(12) + F.col("pickup_month"))
        .withColumn("PULocationID_string", F.col("PULocationID").cast("string"))
        .withColumn("is_weekend_int", F.col("is_weekend").cast("int"))
    )


def select_modeling_dates(demand_df):
    date_df = demand_df.select(
        "pickup_date",
        "pickup_year",
        "pickup_month",
        "pickup_day",
        "pickup_day_of_week",
        "is_weekend",
        "is_weekend_int",
        "month_index",
    ).distinct()

    if DEMO_MODE:
        date_df = date_df.filter(F.pmod(F.dayofyear("pickup_date"), F.lit(DEMO_DATE_STRIDE_DAYS)) == F.lit(1))

    return date_df


def build_complete_location_hour_demand(positive_demand_df):
    locations_df = positive_demand_df.select("PULocationID", "PULocationID_string").distinct()
    dates_df = select_modeling_dates(positive_demand_df)
    hours_df = spark.range(0, 24).select(F.col("id").cast("int").alias("pickup_hour"))

    grid_df = locations_df.crossJoin(dates_df).crossJoin(hours_df)
    demand_values_df = positive_demand_df.select("PULocationID", "pickup_date", "pickup_hour", "trip_count")

    return (
        grid_df
        .join(demand_values_df, ["PULocationID", "pickup_date", "pickup_hour"], "left")
        .withColumn("trip_count", F.coalesce(F.col("trip_count"), F.lit(0.0)))
        .repartition(int(SHUFFLE_PARTITIONS), "pickup_year", "pickup_month")
    )


def build_history_tables(demand_df):
    location_month = (
        demand_df
        .groupBy("PULocationID", "month_index")
        .agg(F.avg("trip_count").alias("month_location_avg"))
    )
    location_window = (
        Window
        .partitionBy("PULocationID")
        .orderBy("month_index")
        .rowsBetween(Window.unboundedPreceding, -1)
    )
    location_history = (
        location_month
        .withColumn("location_avg", F.avg("month_location_avg").over(location_window))
        .select("PULocationID", "month_index", "location_avg")
    )

    location_hour_month = (
        demand_df
        .groupBy("PULocationID", "pickup_hour", "month_index")
        .agg(
            F.avg("trip_count").alias("month_location_hour_avg"),
            F.avg(F.when(F.col("trip_count") == 0, 1.0).otherwise(0.0)).alias("month_location_hour_zero_rate"),
        )
    )
    location_hour_window = (
        Window
        .partitionBy("PULocationID", "pickup_hour")
        .orderBy("month_index")
        .rowsBetween(Window.unboundedPreceding, -1)
    )
    location_hour_history = (
        location_hour_month
        .withColumn("location_hour_avg", F.avg("month_location_hour_avg").over(location_hour_window))
        .withColumn("location_hour_zero_rate", F.avg("month_location_hour_zero_rate").over(location_hour_window))
        .select("PULocationID", "pickup_hour", "month_index", "location_hour_avg", "location_hour_zero_rate")
    )

    location_dow_hour_month = (
        demand_df
        .groupBy("PULocationID", "pickup_day_of_week", "pickup_hour", "month_index")
        .agg(F.avg("trip_count").alias("month_location_dow_hour_avg"))
    )
    location_dow_hour_window = (
        Window
        .partitionBy("PULocationID", "pickup_day_of_week", "pickup_hour")
        .orderBy("month_index")
        .rowsBetween(Window.unboundedPreceding, -1)
    )
    location_dow_hour_history = (
        location_dow_hour_month
        .withColumn("location_dow_hour_avg", F.avg("month_location_dow_hour_avg").over(location_dow_hour_window))
        .select("PULocationID", "pickup_day_of_week", "pickup_hour", "month_index", "location_dow_hour_avg")
    )

    previous_month_location_hour = (
        location_hour_month
        .select(
            "PULocationID",
            "pickup_hour",
            (F.col("month_index") + F.lit(1)).alias("month_index"),
            F.col("month_location_hour_avg").alias("previous_month_location_hour_avg"),
        )
    )

    hour_month = (
        demand_df
        .groupBy("pickup_hour", "month_index")
        .agg(F.avg("trip_count").alias("month_hour_avg"))
    )
    hour_window = (
        Window
        .partitionBy("pickup_hour")
        .orderBy("month_index")
        .rowsBetween(Window.unboundedPreceding, -1)
    )
    hour_history = (
        hour_month
        .withColumn("hour_avg", F.avg("month_hour_avg").over(hour_window))
        .select("pickup_hour", "month_index", "hour_avg")
    )

    return {
        "location_history": location_history,
        "location_hour_history": location_hour_history,
        "location_dow_hour_history": location_dow_hour_history,
        "previous_month_location_hour": previous_month_location_hour,
        "hour_history": hour_history,
    }


def add_historical_features(input_df, history_tables):
    with_history = (
        input_df
        .join(history_tables["location_history"], ["PULocationID", "month_index"], "left")
        .join(history_tables["location_hour_history"], ["PULocationID", "pickup_hour", "month_index"], "left")
        .join(history_tables["location_dow_hour_history"], ["PULocationID", "pickup_day_of_week", "pickup_hour", "month_index"], "left")
        .join(history_tables["previous_month_location_hour"], ["PULocationID", "pickup_hour", "month_index"], "left")
        .join(history_tables["hour_history"], ["pickup_hour", "month_index"], "left")
        .withColumn("location_avg", F.coalesce("location_avg", "hour_avg", F.lit(0.0)))
        .withColumn("location_hour_avg", F.coalesce("location_hour_avg", "location_avg", "hour_avg", F.lit(0.0)))
        .withColumn("location_dow_hour_avg", F.coalesce("location_dow_hour_avg", "location_hour_avg", "location_avg", "hour_avg", F.lit(0.0)))
        .withColumn(
            "previous_month_location_hour_avg",
            F.coalesce("previous_month_location_hour_avg", "location_hour_avg", "location_avg", "hour_avg", F.lit(0.0)),
        )
        .withColumn("location_hour_zero_rate", F.coalesce("location_hour_zero_rate", F.lit(1.0)))
        .drop("hour_avg")
    )
    return with_history


model_input_columns = [
    "PULocationID_string",
    "pickup_month",
    "pickup_day_of_week",
    "pickup_hour",
    "is_weekend_int",
    "location_avg",
    "location_hour_avg",
    "location_dow_hour_avg",
    "previous_month_location_hour_avg",
    "location_hour_zero_rate",
]

positive_demand_df = build_source_agnostic_demand(raw_df).persist(CACHE_STORAGE_LEVEL)

demand_df = build_complete_location_hour_demand(positive_demand_df).persist(CACHE_STORAGE_LEVEL)
history_tables = build_history_tables(demand_df)

dataset = (
    add_historical_features(demand_df, history_tables)
    .withColumn("label", F.col("trip_count"))
    .persist(CACHE_STORAGE_LEVEL)
)

print("Source columns ignored for ML: source_id, source_name")
print("Zero-demand location/date/hour rows are included in the model target.")
print("Model input columns expected before the saved pipeline adds its own encoded features:")
for column in model_input_columns:
    print(f"- {column}")

print("Feature rows are defined. The next cell materializes the 2025 holdout rows.")


## 4. Prepare 2025 Holdout Rows

Local training is no longer part of the notebook. This cell only materializes the 2025 holdout rows used for diagnostics after the GCP model is loaded.


In [ ]:
test_year = 2025
test_df = dataset.filter(F.col("pickup_year") == test_year).persist(CACHE_STORAGE_LEVEL)

test_count = test_df.count()
if test_count == 0:
    raise ValueError(f"No test rows found for year: {test_year}")

print(f"Test year: {test_year}")
print(f"2025 holdout rows: {test_count:,}")
print("Training is skipped locally. The next section reads the downloaded GCP artifacts.")


## 5. Read GCP Training Artifacts

This section reads the downloaded `metrics.csv`, selects the best trainable row by RMSE, and loads that row's saved `model_artifact` path in Step 7. Use `GCP_RUN_ID` in Step 1 when you want a specific run; otherwise the newest complete artifact folder is selected.


In [ ]:
def resolve_gcp_training_output_dir():
    if GCP_RUN_ID:
        candidate = GCP_TRAINING_ROOT / f"gcp_training_{GCP_RUN_ID}"
        if not candidate.exists():
            raise FileNotFoundError(f"Configured GCP artifact directory does not exist: {candidate}")
        return candidate

    candidates = sorted(
        GCP_TRAINING_ROOT.glob("gcp_training_*"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    for candidate in candidates:
        required_paths = [
            candidate / "metrics.csv",
            candidate / "models",
        ]
        if all(path.exists() for path in required_paths):
            return candidate

    raise FileNotFoundError(
        "No downloaded GCP training artifact directory found. Expected metrics.csv "
        "and models/ under output/gcp_training_<run_id>/."
    )


gcp_training_output_dir = resolve_gcp_training_output_dir()
gcp_metrics_path = gcp_training_output_dir / "metrics.csv"

metrics_pdf = pd.read_csv(gcp_metrics_path)
metrics_pdf["rmse"] = metrics_pdf["rmse"].astype(float)
metrics_pdf["mae"] = metrics_pdf["mae"].astype(float)
metrics_pdf["r2"] = metrics_pdf["r2"].astype(float)
metrics_pdf["row_count"] = metrics_pdf["row_count"].astype(int)
if "train_row_count" in metrics_pdf.columns:
    metrics_pdf["train_row_count"] = metrics_pdf["train_row_count"].fillna(0).astype(int)
if "model_artifact" not in metrics_pdf.columns:
    raise ValueError("metrics.csv must include a model_artifact column for CSV-based model loading")

metrics_pdf["model_artifact"] = metrics_pdf["model_artifact"].fillna("").astype(str)
metrics_df = spark.createDataFrame(metrics_pdf)

trainable_metrics_pdf = metrics_pdf[
    (metrics_pdf["selection"] == "trainable")
    & (metrics_pdf["model_artifact"].str.strip() != "")
].sort_values("rmse")
if trainable_metrics_pdf.empty:
    raise ValueError("No trainable rows with model_artifact were found in metrics.csv")

best_row = trainable_metrics_pdf.iloc[0]
best_model_name = best_row["model"]
best_model_config = {
    "config_name": best_row["config"],
    "params_text": best_row["params"],
}
best_result = {
    "model": best_model_name,
    "config": best_model_config["config_name"],
    "params": best_model_config["params_text"],
    "rmse": float(best_row["rmse"]),
    "mae": float(best_row["mae"]),
    "r2": float(best_row["r2"]),
    "row_count": int(best_row["row_count"]),
    "selection": best_row.get("selection", "trainable"),
    "train_row_count": int(best_row.get("train_row_count", 0) or 0),
    "model_artifact": best_row["model_artifact"],
}
best_model_output_path = str(gcp_training_output_dir / best_result["model_artifact"])

if not Path(best_model_output_path).exists():
    raise FileNotFoundError(f"Selected model artifact does not exist: {best_model_output_path}")

print(f"Loaded GCP metrics from: {gcp_metrics_path}")
print(f"Selected model artifact from metrics.csv: {best_model_output_path}")
metrics_df.orderBy("rmse").show(truncate=False)

print(f"""
Best Trainable Model: {best_model_name}
Best Config: {best_model_config['config_name']}
Params: {best_model_config['params_text']}
RMSE: {best_result['rmse']:.4f}
MAE: {best_result['mae']:.4f}
R2: {best_result['r2']:.4f}
Rows Evaluated: {best_result['row_count']:,}
Training Rows Used: {best_result['train_row_count']:,}
Model Artifact: {best_result['model_artifact']}
""")


## 6. Model Accuracy Plot

This plot compares the baseline and GCP-tuned trainable model configurations on the 2025 holdout set.


In [ ]:
import matplotlib.pyplot as plt

metrics_pdf = (
    metrics_df
    .where(F.col("selection") == "trainable")
    .orderBy("rmse")
    .toPandas()
)
metrics_pdf["display_name"] = metrics_pdf["model"] + " (" + metrics_pdf["config"] + ")"

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].barh(metrics_pdf["display_name"], metrics_pdf["rmse"], color="#2563eb")
axes[0].invert_yaxis()
axes[0].set_title("GCP Model Comparison by RMSE")
axes[0].set_xlabel("RMSE, lower is better")

axes[1].barh(metrics_pdf["display_name"], metrics_pdf["mae"], color="#16a34a")
axes[1].invert_yaxis()
axes[1].set_title("GCP Model Comparison by MAE")
axes[1].set_xlabel("MAE, lower is better")

plt.tight_layout()
plt.show()


## 7. Load the Best GCP Model

The saved GCP pipeline already contains preprocessing and the selected estimator. After this cell, the remaining diagnostics can call `best_pipeline_model.transform(...)` directly on the feature rows built above.

If the best model is XGBoost, the notebook kernel must have the matching `xgboost` Python package installed before this cell runs.


In [ ]:
def require_xgboost_for_saved_model(model_name):
    if model_name != "SparkXGBRegressor":
        return

    try:
        import xgboost  # noqa: F401
        from xgboost.spark import SparkXGBRegressorModel  # noqa: F401
    except ImportError as exc:
        raise ImportError(
            "The saved GCP best model is SparkXGBRegressor, so this notebook kernel needs "
            "xgboost==2.1.4 installed before loading best_pipeline_model. Install it in the "
            "same environment that runs this notebook, restart the kernel, then rerun from Step 1."
        ) from exc


require_xgboost_for_saved_model(best_model_name)

best_pipeline_model = PipelineModel.load(best_model_output_path)

print(f"Best pipeline model loaded from: {best_model_output_path}")
print(f"""
Best Trainable Model: {best_model_name}
Best Config: {best_model_config['config_name']}
Params: {best_model_config['params_text']}
RMSE: {best_result['rmse']:.4f}
MAE: {best_result['mae']:.4f}
R2: {best_result['r2']:.4f}
""")

gc.collect()


## 8. Actual vs Predicted Comparison

These diagnostics compare the selected trained model's predictions with the actual 2025 demand counts. The residual is `predicted_trip_count - actual_trip_count`, so positive values mean the model overpredicted demand.

Read the plots carefully: the scatter plot is row-level accuracy for each pickup location/date/hour row, while the hourly line chart is averaged across all rows with the same hour. A model can match the average hourly curve but still miss many individual high-demand or low-demand location rows.

In [ ]:
Path("output").mkdir(exist_ok=True)

try:
    test_predictions.unpersist()
except NameError:
    pass

test_predictions = (
    best_pipeline_model
    .transform(test_df)
    .select(
        "PULocationID",
        "pickup_date",
        "pickup_year",
        "pickup_month",
        "pickup_day",
        "pickup_day_of_week",
        "pickup_hour",
        "is_weekend",
        "location_avg",
        "location_hour_avg",
        "location_dow_hour_avg",
        "previous_month_location_hour_avg",
        "location_hour_zero_rate",
        F.col("trip_count").alias("actual_trip_count"),
        F.col("prediction").alias("predicted_trip_count"),
    )
    .withColumn("prediction_error", F.col("predicted_trip_count") - F.col("actual_trip_count"))
    .withColumn("absolute_error", F.abs(F.col("prediction_error")))
    .persist(CACHE_STORAGE_LEVEL)
)

test_prediction_count = test_predictions.count()
print(f"2025 prediction rows: {test_prediction_count:,}")

sample_predictions = (
    test_predictions
    .select(
        "PULocationID",
        "pickup_date",
        "pickup_hour",
        F.round("actual_trip_count", 2).alias("actual_trip_count"),
        F.round("predicted_trip_count", 2).alias("predicted_trip_count"),
        F.round("prediction_error", 2).alias("prediction_error"),
        F.round("absolute_error", 2).alias("absolute_error"),
    )
    .orderBy("PULocationID", "pickup_date", "pickup_hour")
)

sample_predictions.show(10, truncate=False)
sample_predictions.limit(1000).toPandas().to_csv("output/sample_predictions.csv", index=False)
print("Sample predictions saved to output/sample_predictions.csv")


In [ ]:
prediction_summary = (
    test_predictions
    .agg(
        F.count("*").alias("rows"),
        F.round(F.avg("actual_trip_count"), 2).alias("avg_actual_trip_count"),
        F.round(F.avg("predicted_trip_count"), 2).alias("avg_predicted_trip_count"),
        F.round(F.avg("prediction_error"), 2).alias("mean_error"),
        F.round(F.avg("absolute_error"), 2).alias("mean_absolute_error"),
        F.round(F.sqrt(F.avg(F.col("prediction_error") * F.col("prediction_error"))), 2).alias("rmse"),
        F.round(F.corr("actual_trip_count", "predicted_trip_count"), 4).alias("actual_prediction_corr"),
    )
)

prediction_summary.show(truncate=False)

zero_actual_summary = (
    test_predictions
    .filter(F.col("actual_trip_count") == 0)
    .agg(
        F.count("*").alias("zero_actual_rows"),
        F.round(F.avg("predicted_trip_count"), 2).alias("avg_prediction_when_actual_zero"),
        F.round(F.max("predicted_trip_count"), 2).alias("max_prediction_when_actual_zero"),
        F.round(F.avg(F.when(F.col("predicted_trip_count") >= 20, 1.0).otherwise(0.0)) * 100.0, 2).alias("pct_zero_rows_predicted_20_plus"),
    )
)

print("Zero-actual rows should now be common because the notebook creates the full location/date/hour grid.")
zero_actual_summary.show(truncate=False)

actual_bucketed_errors = (
    test_predictions
    .withColumn(
        "actual_bucket_order",
        F.when(F.col("actual_trip_count") < 10, 1)
         .when(F.col("actual_trip_count") < 50, 2)
         .when(F.col("actual_trip_count") < 100, 3)
         .when(F.col("actual_trip_count") < 250, 4)
         .otherwise(5),
    )
    .withColumn(
        "actual_trip_count_bucket",
        F.when(F.col("actual_trip_count") < 10, "0-9")
         .when(F.col("actual_trip_count") < 50, "10-49")
         .when(F.col("actual_trip_count") < 100, "50-99")
         .when(F.col("actual_trip_count") < 250, "100-249")
         .otherwise("250+"),
    )
    .groupBy("actual_bucket_order", "actual_trip_count_bucket")
    .agg(
        F.count("*").alias("rows"),
        F.round(F.avg("actual_trip_count"), 2).alias("avg_actual"),
        F.round(F.avg("predicted_trip_count"), 2).alias("avg_predicted"),
        F.round(F.avg("prediction_error"), 2).alias("mean_error"),
        F.round(F.avg("absolute_error"), 2).alias("mae"),
        F.round(F.sqrt(F.avg(F.col("prediction_error") * F.col("prediction_error"))), 2).alias("rmse"),
    )
    .orderBy("actual_bucket_order")
    .drop("actual_bucket_order")
)

print("Error by actual demand bucket. This shows whether high-demand rows are being underpredicted.")
actual_bucketed_errors.show(truncate=False)

worst_location_errors = (
    test_predictions
    .groupBy("PULocationID")
    .agg(
        F.count("*").alias("rows"),
        F.round(F.avg("actual_trip_count"), 2).alias("avg_actual"),
        F.round(F.avg("predicted_trip_count"), 2).alias("avg_predicted"),
        F.round(F.avg("absolute_error"), 2).alias("mae"),
    )
    .filter(F.col("rows") >= 20)
    .orderBy(F.desc("mae"))
)

print("Pickup locations with the largest average errors.")
worst_location_errors.show(10, truncate=False)

largest_errors = (
    test_predictions
    .select(
        "PULocationID",
        "pickup_date",
        "pickup_hour",
        F.round("actual_trip_count", 2).alias("actual_trip_count"),
        F.round("predicted_trip_count", 2).alias("predicted_trip_count"),
        F.round("prediction_error", 2).alias("prediction_error"),
        F.round("absolute_error", 2).alias("absolute_error"),
    )
    .orderBy(F.desc("absolute_error"))
)

print("Largest individual row errors.")
largest_errors.show(10, truncate=False)


## Pickup Location Accuracy Ranking

This chart ranks pickup locations by final 2025 prediction error. Lower MAE means better location-level accuracy. The full ranking is also saved as CSV for review.


In [ ]:
import matplotlib.pyplot as plt

Path("output").mkdir(exist_ok=True)

location_accuracy_df = (
    test_predictions
    .groupBy("PULocationID")
    .agg(
        F.count("*").alias("rows"),
        F.avg("actual_trip_count").alias("avg_actual"),
        F.avg("predicted_trip_count").alias("avg_predicted"),
        F.avg("absolute_error").alias("mae"),
        F.sqrt(F.avg(F.col("prediction_error") * F.col("prediction_error"))).alias("rmse"),
        F.avg("prediction_error").alias("bias"),
    )
    .join(zone_lookup_df.select("PULocationID", "borough", "zone"), "PULocationID", "left")
    .withColumn(
        "accuracy_rank",
        F.row_number().over(Window.orderBy(F.asc("mae"), F.asc("rmse"), F.asc("PULocationID"))),
    )
    .select(
        "accuracy_rank",
        "PULocationID",
        "borough",
        "zone",
        "rows",
        F.round("avg_actual", 2).alias("avg_actual"),
        F.round("avg_predicted", 2).alias("avg_predicted"),
        F.round("mae", 2).alias("mae"),
        F.round("rmse", 2).alias("rmse"),
        F.round("bias", 2).alias("bias"),
    )
    .orderBy("accuracy_rank")
)

location_accuracy_df.show(20, truncate=False)

location_accuracy_pdf = location_accuracy_df.toPandas()
location_accuracy_pdf.to_csv("output/location_accuracy_ranked.csv", index=False)
print("Location accuracy ranking saved to output/location_accuracy_ranked.csv")

if location_accuracy_pdf.empty:
    print("No rows available for location accuracy ranking plot.")
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(
        location_accuracy_pdf["accuracy_rank"],
        location_accuracy_pdf["mae"],
        marker=".",
        linewidth=1,
    )
    ax.set_title("Pickup Location Accuracy Ranking on 2025 Holdout")
    ax.set_xlabel("Location rank, lower MAE is better")
    ax.set_ylabel("MAE by pickup location")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    worst_location_pdf = location_accuracy_pdf.sort_values("mae", ascending=False).head(20).copy()
    worst_location_pdf["location_label"] = (
        worst_location_pdf["PULocationID"].astype(str)
        + " - "
        + worst_location_pdf["zone"].fillna("Unknown")
    )
    worst_location_pdf = worst_location_pdf.sort_values("mae")

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(worst_location_pdf["location_label"], worst_location_pdf["mae"], color="#dc2626")
    ax.set_title("Worst Pickup Locations by 2025 MAE")
    ax.set_xlabel("MAE, lower is better")
    ax.set_ylabel("Pickup location")
    plt.tight_layout()
    plt.show()


In [ ]:
comparison_sample_pdf = (
    test_predictions
    .sample(False, PLOT_SAMPLE_FRACTION, seed=42)
    .limit(PLOT_SAMPLE_LIMIT)
    .select("actual_trip_count", "predicted_trip_count")
    .toPandas()
)

if comparison_sample_pdf.empty:
    comparison_sample_pdf = (
        test_predictions
        .limit(PLOT_SAMPLE_LIMIT)
        .select("actual_trip_count", "predicted_trip_count")
        .toPandas()
    )

if comparison_sample_pdf.empty:
    print("No rows available for actual vs predicted plot.")
else:
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(
        comparison_sample_pdf["actual_trip_count"],
        comparison_sample_pdf["predicted_trip_count"],
        alpha=0.25,
        s=12,
    )

    max_axis = max(
        comparison_sample_pdf["actual_trip_count"].max(),
        comparison_sample_pdf["predicted_trip_count"].max(),
        1,
    )
    ax.plot([0, max_axis], [0, max_axis], color="#dc2626", linestyle="--", linewidth=1, label="perfect prediction")
    ax.set_title("Row-Level Actual vs Predicted 2025 Trip Count")
    ax.set_xlabel("Actual trip count per pickup location/date/hour")
    ax.set_ylabel("Predicted trip count per pickup location/date/hour")
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
hourly_comparison_pdf = (
    test_predictions
    .groupBy("pickup_hour")
    .agg(
        F.avg("actual_trip_count").alias("avg_actual_trip_count"),
        F.avg("predicted_trip_count").alias("avg_predicted_trip_count"),
        F.avg("prediction_error").alias("avg_prediction_error"),
    )
    .orderBy("pickup_hour")
    .toPandas()
)

if hourly_comparison_pdf.empty:
    print("No rows available for hourly actual vs predicted plot.")
else:
    print("This hourly plot is averaged across all pickup locations and dates. Use it for time-of-day trend only; use the scatter and error buckets for row-level accuracy.")

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(hourly_comparison_pdf["pickup_hour"], hourly_comparison_pdf["avg_actual_trip_count"], marker="o", label="Actual")
    ax.plot(hourly_comparison_pdf["pickup_hour"], hourly_comparison_pdf["avg_predicted_trip_count"], marker="o", label="Predicted")
    ax.set_title("Hourly Average Actual vs Predicted Demand")
    ax.set_xlabel("Pickup hour")
    ax.set_ylabel("Average trip count across all rows")
    ax.set_xticks(range(0, 24, 2))
    ax.legend()
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(hourly_comparison_pdf["pickup_hour"], hourly_comparison_pdf["avg_prediction_error"], marker="o")
    ax.axhline(0, color="#111827", linewidth=1)
    ax.set_title("Average Prediction Error by Pickup Hour")
    ax.set_xlabel("Pickup hour")
    ax.set_ylabel("Predicted minus actual trips")
    ax.set_xticks(range(0, 24, 2))
    plt.tight_layout()
    plt.show()

residual_pdf = (
    test_predictions
    .sample(False, PLOT_SAMPLE_FRACTION, seed=24)
    .limit(PLOT_SAMPLE_LIMIT)
    .select("prediction_error")
    .toPandas()
)

if not residual_pdf.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(residual_pdf["prediction_error"], bins=40, alpha=0.75)
    ax.axvline(0, color="#111827", linewidth=1)
    ax.set_title("Residual Distribution on 2025 Holdout")
    ax.set_xlabel("Predicted minus actual trips")
    ax.set_ylabel("Frequency")
    plt.tight_layout()
    plt.show()


## 9. Save Streamlit Prediction Output

Streamlit should not train the model. This section first keeps the selected-month prediction shape used earlier, then the added refit cell replaces `serving_predictions` with a full-year 2026 forecast trained on all 2023-2025 rows before the export is written.


In [ ]:
serving_month_index = SERVING_PREDICTION_YEAR * 12 + SERVING_PREDICTION_MONTH

pickup_locations_df = demand_df.select("PULocationID", "PULocationID_string").distinct()
day_of_week_df = spark.range(1, 8).select(F.col("id").cast("int").alias("pickup_day_of_week"))
hour_df = spark.range(0, 24).select(F.col("id").cast("int").alias("pickup_hour"))

serving_grid_df = (
    pickup_locations_df
    .crossJoin(day_of_week_df)
    .crossJoin(hour_df)
    .withColumn("pickup_year", F.lit(SERVING_PREDICTION_YEAR).cast("int"))
    .withColumn("pickup_month", F.lit(SERVING_PREDICTION_MONTH).cast("int"))
    .withColumn("pickup_day", F.lit(1).cast("int"))
    .withColumn("pickup_date", date_from_year_month_day(F.col("pickup_year"), F.col("pickup_month"), F.col("pickup_day")))
    .withColumn("month_index", F.lit(serving_month_index).cast("int"))
    .withColumn("is_weekend", F.col("pickup_day_of_week").isin([1, 7]))
    .withColumn("is_weekend_int", F.col("is_weekend").cast("int"))
)

serving_dataset = add_historical_features(serving_grid_df, history_tables)

serving_predictions = (
    best_pipeline_model
    .transform(serving_dataset)
    .withColumn("predicted_trip_count", F.greatest(F.col("prediction"), F.lit(0.0)))
    .join(zone_lookup_df, "PULocationID", "left")
    .withColumn("prediction_timestamp", F.current_timestamp())
    .withColumn(
        "demand_level",
        F.when(F.col("predicted_trip_count") < 20, "low")
         .when(F.col("predicted_trip_count") < 50, "medium")
         .when(F.col("predicted_trip_count") < 100, "high")
         .otherwise("very_high"),
    )
    .select(
        "prediction_timestamp",
        "PULocationID",
        "borough",
        "zone",
        "pickup_year",
        "pickup_month",
        "pickup_day_of_week",
        "pickup_hour",
        F.round("predicted_trip_count", 2).alias("predicted_trip_count"),
        "demand_level",
    )
)

print(f"Serving prediction month: {SERVING_PREDICTION_YEAR}-{SERVING_PREDICTION_MONTH:02d}")
print(f"Serving prediction rows: {serving_predictions.count():,}")
serving_predictions.orderBy("pickup_hour", F.desc("predicted_trip_count")).show(10, truncate=False)


In [ ]:
# Refit a lightweight serving model on all known 2023-2025 rows, then forecast every Streamlit month in 2026.
from pyspark.ml import Pipeline
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression

FORECAST_TRAIN_YEARS = [2023, 2024, 2025]
FORECAST_YEAR = 2026
FORECAST_MONTHS = list(range(1, 13))

forecast_train_df = (
    dataset
    .filter(F.col("pickup_year").isin(FORECAST_TRAIN_YEARS))
    .persist(CACHE_STORAGE_LEVEL)
)
forecast_train_count = forecast_train_df.count()
if forecast_train_count == 0:
    raise ValueError(f"No training rows found for years: {FORECAST_TRAIN_YEARS}")

forecast_numeric_feature_cols = [
    "pickup_month",
    "pickup_day_of_week",
    "pickup_hour",
    "is_weekend_int",
    "location_avg",
    "location_hour_avg",
    "location_dow_hour_avg",
    "previous_month_location_hour_avg",
    "location_hour_zero_rate",
]

forecast_pipeline = Pipeline(stages=[
    StringIndexer(inputCol="PULocationID_string", outputCol="PULocationID_index", handleInvalid="keep"),
    OneHotEncoder(inputCols=["PULocationID_index"], outputCols=["PULocationID_ohe"], handleInvalid="keep"),
    VectorAssembler(inputCols=forecast_numeric_feature_cols + ["PULocationID_ohe"], outputCol="features", handleInvalid="skip"),
    LinearRegression(
        featuresCol="features",
        labelCol="label",
        predictionCol="prediction",
        maxIter=30,
        regParam=0.0,
        elasticNetParam=0.0,
    ),
])

forecast_pipeline_model = forecast_pipeline.fit(forecast_train_df)

# Future rows need history features. For 2026, use all completed 2023-2025 demand history
# instead of joining by exact future month_index, which would otherwise produce empty features.
full_history_df = (
    demand_df
    .filter(F.col("pickup_year").isin(FORECAST_TRAIN_YEARS))
    .persist(CACHE_STORAGE_LEVEL)
)

future_location_history = (
    full_history_df
    .groupBy("PULocationID")
    .agg(F.avg("trip_count").alias("location_avg"))
)
future_location_hour_history = (
    full_history_df
    .groupBy("PULocationID", "pickup_hour")
    .agg(
        F.avg("trip_count").alias("location_hour_avg"),
        F.avg(F.when(F.col("trip_count") == 0, 1.0).otherwise(0.0)).alias("location_hour_zero_rate"),
    )
)
future_location_dow_hour_history = (
    full_history_df
    .groupBy("PULocationID", "pickup_day_of_week", "pickup_hour")
    .agg(F.avg("trip_count").alias("location_dow_hour_avg"))
)
future_previous_month_history = (
    full_history_df
    .withColumn("previous_calendar_month", F.col("pickup_month"))
    .groupBy("PULocationID", "pickup_hour", "previous_calendar_month")
    .agg(F.avg("trip_count").alias("previous_month_location_hour_avg"))
)

pickup_locations_df = demand_df.select("PULocationID", "PULocationID_string").distinct()
month_df = spark.range(1, 13).select(F.col("id").cast("int").alias("pickup_month"))
day_of_week_df = spark.range(1, 8).select(F.col("id").cast("int").alias("pickup_day_of_week"))
hour_df = spark.range(0, 24).select(F.col("id").cast("int").alias("pickup_hour"))

serving_grid_df = (
    pickup_locations_df
    .crossJoin(month_df)
    .crossJoin(day_of_week_df)
    .crossJoin(hour_df)
    .withColumn("pickup_year", F.lit(FORECAST_YEAR).cast("int"))
    .withColumn("pickup_day", F.lit(1).cast("int"))
    .withColumn("pickup_date", date_from_year_month_day(F.col("pickup_year"), F.col("pickup_month"), F.col("pickup_day")))
    .withColumn("month_index", F.col("pickup_year") * F.lit(12) + F.col("pickup_month"))
    .withColumn("previous_calendar_month", F.when(F.col("pickup_month") == 1, F.lit(12)).otherwise(F.col("pickup_month") - 1))
    .withColumn("is_weekend", F.col("pickup_day_of_week").isin([1, 7]))
    .withColumn("is_weekend_int", F.col("is_weekend").cast("int"))
)

serving_dataset = (
    serving_grid_df
    .join(future_location_history, ["PULocationID"], "left")
    .join(future_location_hour_history, ["PULocationID", "pickup_hour"], "left")
    .join(future_location_dow_hour_history, ["PULocationID", "pickup_day_of_week", "pickup_hour"], "left")
    .join(future_previous_month_history, ["PULocationID", "pickup_hour", "previous_calendar_month"], "left")
    .withColumn("location_avg", F.coalesce("location_avg", F.lit(0.0)))
    .withColumn("location_hour_avg", F.coalesce("location_hour_avg", "location_avg", F.lit(0.0)))
    .withColumn("location_dow_hour_avg", F.coalesce("location_dow_hour_avg", "location_hour_avg", "location_avg", F.lit(0.0)))
    .withColumn(
        "previous_month_location_hour_avg",
        F.coalesce("previous_month_location_hour_avg", "location_hour_avg", "location_avg", F.lit(0.0)),
    )
    .withColumn("location_hour_zero_rate", F.coalesce("location_hour_zero_rate", F.lit(1.0)))
    .drop("previous_calendar_month")
)

serving_predictions = (
    forecast_pipeline_model
    .transform(serving_dataset)
    .withColumn("predicted_trip_count", F.greatest(F.col("prediction"), F.lit(0.0)))
    .join(zone_lookup_df, "PULocationID", "left")
    .withColumn("prediction_timestamp", F.current_timestamp())
    .withColumn(
        "demand_level",
        F.when(F.col("predicted_trip_count") < 20, "low")
         .when(F.col("predicted_trip_count") < 50, "medium")
         .when(F.col("predicted_trip_count") < 100, "high")
         .otherwise("very_high"),
    )
    .select(
        "prediction_timestamp",
        "PULocationID",
        "borough",
        "zone",
        "pickup_year",
        "pickup_month",
        "pickup_day_of_week",
        "pickup_hour",
        F.round("predicted_trip_count", 2).alias("predicted_trip_count"),
        "demand_level",
    )
    .persist(CACHE_STORAGE_LEVEL)
)

forecast_row_count = serving_predictions.count()
print(f"Forecast training years: {FORECAST_TRAIN_YEARS}")
print(f"Forecast training rows: {forecast_train_count:,}")
print(f"Streamlit forecast year: {FORECAST_YEAR}")
print(f"Streamlit forecast months: {FORECAST_MONTHS[0]}-{FORECAST_MONTHS[-1]}")
print(f"Streamlit forecast rows: {forecast_row_count:,}")
serving_predictions.orderBy("pickup_month", "pickup_hour", F.desc("predicted_trip_count")).show(10, truncate=False)


In [ ]:
# Average predicted demand by pickup hour for Streamlit.
hourly_serving_pdf = (
    serving_predictions
    .groupBy("pickup_hour")
    .agg(F.avg("predicted_trip_count").alias("avg_predicted_trip_count"))
    .orderBy("pickup_hour")
    .toPandas()
)

plt.figure(figsize=(9, 5))
plt.plot(hourly_serving_pdf["pickup_hour"], hourly_serving_pdf["avg_predicted_trip_count"], marker="o")
plt.title("Average Streamlit Prediction by Hour")
plt.xlabel("Hour of day")
plt.ylabel("Average predicted trip count")
plt.xticks(range(0, 24, 2))
plt.tight_layout()
plt.show()


In [ ]:
Path("output").mkdir(exist_ok=True)
streamlit_predictions_path = "output/streamlit_predictions"
streamlit_predictions_csv_path = "output/streamlit_predictions_csv"

serving_predictions.write.mode("overwrite").parquet(streamlit_predictions_path)
(
    serving_predictions
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(streamlit_predictions_csv_path)
)

print(f"Streamlit predictions saved to {streamlit_predictions_path}")
print(f"Streamlit CSV export saved to {streamlit_predictions_csv_path}")


## 10. Stop Spark

Stop the Spark session when the demo is finished so local resources are released.


In [ ]:
for cached_name in [
    "positive_demand_df",
    "demand_df",
    "dataset",
    "test_df",
    "test_predictions",
    "forecast_train_df",
    "full_history_df",
    "serving_predictions",
]:
    cached_df = globals().get(cached_name)
    if cached_df is not None:
        cached_df.unpersist()

spark.stop()
